In [25]:
import pandas as pd
df=pd.read_csv('../data/processed/final_df.csv', index_col=0)
df

,transformed_text_title,type
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0
...,...,...
44684,BRUSSELS (Reuters) - NATO allies on Tuesday we...,1
44685,"LONDON (Reuters) - LexisNexis, a provider of l...",1
44686,MINSK (Reuters) - In the shadow of disused Sov...,1
44687,MOSCOW (Reuters) - Vatican Secretary of State ...,1


In [27]:
from sentence_transformers import SentenceTransformer

embedder=SentenceTransformer('all-MiniLM-L6-v2')

X=embedder.encode(df['transformed_text_title'].tolist(),batch_size=128,show_progress_bar=True)

print('shape of features', X.shape)

y=df['type']


Batches: 100%|██████████| 350/350 [01:14<00:00,  4.72it/s]


shape of features (44689, 384)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
models = {
    "log_reg": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "svm": LinearSVC(dual=False, max_iter=1000, random_state=42),
    "rf": RandomForestClassifier(n_estimators=400, max_depth=8, random_state=42, n_jobs=-1),
    "xgb": XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, eval_metric="logloss"),
    "lgbm":LGBMClassifier(n_estimators=400, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
}

In [32]:
for name, model in models.items():
    print (f'Training {name}...')
    model.fit(X_train,y_train)

    y_pred=model.predict(X_test)

    acc=accuracy_score(y_test, y_pred)*100
    print(f'{name} accuracy: {acc:2f}%')
    print(classification_report(y_test, y_pred, target_names=['Fake(0)','Real(1)']))
    print ('-' * 60)

Training log_reg...


c:\Users\aasud\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


log_reg accuracy: 96.554039%
              precision    recall  f1-score   support

     Fake(0)       0.97      0.96      0.97      4696
     Real(1)       0.96      0.97      0.96      4242

    accuracy                           0.97      8938
   macro avg       0.97      0.97      0.97      8938
weighted avg       0.97      0.97      0.97      8938

------------------------------------------------------------
Training svm...
svm accuracy: 97.527411%
              precision    recall  f1-score   support

     Fake(0)       0.98      0.97      0.98      4696
     Real(1)       0.97      0.98      0.97      4242

    accuracy                           0.98      8938
   macro avg       0.98      0.98      0.98      8938
weighted avg       0.98      0.98      0.98      8938

------------------------------------------------------------
Training rf...
rf accuracy: 89.080331%
              precision    recall  f1-score   support

     Fake(0)       0.88      0.91      0.90      4696
     R

c:\Users\aasud\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [33]:
from sklearn.ensemble import VotingClassifier

voting_clf=VotingClassifier(
    estimators=[
        ('linear_svc', LinearSVC(dual=False, max_iter=1000, random_state=42, C=1.0)),
        ('xgb', XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, eval_metric='logloss',use_label_encoder=False)),
        ('lgbm',LGBMClassifier(n_estimators=400, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1, verbose=-1))
    ],
    voting='hard',
    n_jobs=-1
)

print ('Training voting ensemble (best 3 models)...')
voting_clf.fit(X_train,y_train)

y_pred_voting=voting_clf.predict(X_test)
acc_voting= accuracy_score(y_test,y_pred_voting)*100

print(f'voting ensemble accuracy: {acc_voting:2f}%')
print(classification_report(y_test, y_pred_voting, target_names=['Fake(0)','Real(1)']))

Training voting ensemble (best 3 models)...
voting ensemble accuracy: 97.874245%
              precision    recall  f1-score   support

     Fake(0)       0.98      0.98      0.98      4696
     Real(1)       0.98      0.98      0.98      4242

    accuracy                           0.98      8938
   macro avg       0.98      0.98      0.98      8938
weighted avg       0.98      0.98      0.98      8938



c:\Users\aasud\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [35]:
#Hyperparameter tuning 

from sklearn.model_selection  import RandomizedSearchCV
import numpy as np

param_dist={
    'C': np.logspace(-3,2,20),
    'max_iter':[1000,2000],
    'tol':[1e-4, 1e-3]
}

search=RandomizedSearchCV(
    LinearSVC(dual=False, random_state=42),
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train,y_train)

print(f'Best parameters: {search.best_params_}')
print(f'Best CV accuracy: {search.best_score_:.4f}')

best_svc=search.best_estimator_
y_pred_tuned=best_svc.predict(X_test)

acc_tuned=accuracy_score(y_test, y_pred_tuned)*100
print (f'Tuned LinearSVC Accuracy:{acc_tuned:.2f}%')
print(classification_report(y_test, y_pred_tuned, target_names=['Fake(0)','Real(1)']))


Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best parameters: {'tol': 0.0001, 'max_iter': 2000, 'C': np.float64(29.763514416313193)}
Best CV accuracy: 0.9731
Tuned LinearSVC Accuracy:97.59%
              precision    recall  f1-score   support

     Fake(0)       0.98      0.98      0.98      4696
     Real(1)       0.97      0.98      0.97      4242

    accuracy                           0.98      8938
   macro avg       0.98      0.98      0.98      8938
weighted avg       0.98      0.98      0.98      8938



In [36]:
import joblib


## saving voting ensemble best model

joblib.dump(voting_clf, '../models/model.pkl')

## saving sentence transformer
embedder=SentenceTransformer('all-MiniLM-L6-v2')
embedder.save('../models/sentence_transformer_model')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.79it/s]
